# Time Series Analysis

In [1]:
import numpy as np

import sys
sys.path.append('../src')
from helpers import column_utils, csv_utils, plot_utils

In [2]:
df = csv_utils.csv_read('../data/cleaned/merged_ras_base_right_join.csv', ',')

***

## Filtering 

We will apply some filters to get the data we want to plot. 

- First we will filter out any of the Special Team positions, these include Long Snapper (LS) and Place Kicker (PK)
    - These positions tend to have very low RAS scores, and they are on the field a shorter amount of time compared to other positions
- Next, we will filter out any Undrafted players, since this analysis is only looking at NFL Draft activity
- Let's also filter out any players without a RAS score. We do not want the median RAS calculation to be skewed by players with missing scores.

In [3]:
df_filtered = df[~df['POS'].isin(['ST', 'LS', 'PK'])]


df_filtered = df_filtered[df_filtered['Round'].notna()]


df_filtered = df_filtered[df_filtered['RAS'].notna()]

## Combining Positions

There are many positions represented within our dataset. In order to cut down on the number of lines we will plot, we will group specific position under its general position category. As an example, a center (OC) will have its positioned changed to offensive lineman (OL)

- Defensive Lineman (DL)
    - Includes DT and DE  
- Offnsive Lineman (OL)
    - OC, OG, OT
- Defensive Back (DB)
    - S, SS, FS, CB
- Running Back (RB)
    - FB

In [4]:
df_filtered['POS'] = column_utils.column_replace_value(
    df_filtered['POS'], column_utils.DEFENSIVE_BACKS, 'DB'
)


df_filtered['POS'] = column_utils.column_replace_value(
    df_filtered['POS'], column_utils.DEFENSIVE_LINEMAN, 'DL'
)


df_filtered['POS'] = column_utils.column_replace_value(
    df_filtered['POS'], column_utils.OFFENSIVE_LINEMAN, 'OL'
)


df_filtered['POS'] = column_utils.column_replace_value(
    df_filtered['POS'], column_utils.RUNNING_BACKS, 'RB'
)

***

## Creating The DataFrames using `groupby`

In [5]:
df_ras_position = df_filtered.groupby(
    ['Year', 'POS']
)['RAS'].median().reset_index()


df_ras_round = df_filtered.groupby(
    ['Year', 'Round']
)['RAS'].median().reset_index()

For the x axis tick marks, lets use every year value from our dataset

For the y axis tick marks, we can use the RAS values from our dataset to get a range from lowest to highest with evenly spaced intervals

In [6]:
years = sorted(df_filtered['Year'].unique())[-1::-2]


years.sort()


scores_positions = np.arange(
    round(df_ras_position.RAS.min() * 2) / 2, 10.5, step=0.5
)


scores_rounds = np.arange(
    round(df_ras_round.RAS.min() * 2) / 2, 10.5, step=0.5
)

***

## Finally lets plot our Time Series

Each time series will be accompanied by the variance of the median RAS scores for each position/draft round per year

In [7]:
plot_utils.interactive_line_plot_checkboxes(df_ras_position, "POS", years, scores_positions)

Label(value='Select Positions:')

GridBox(children=(Checkbox(value=True, description='DB'), Checkbox(value=True, description='DL'), Checkbox(val…

Output()

In [8]:
plot_utils.interactive_line_plot_checkboxes(df_ras_round, "Round", years, scores_rounds)

Label(value='Select Draft Rounds:')

GridBox(children=(Checkbox(value=True, description='1'), Checkbox(value=True, description='2'), Checkbox(value…

Output()